# Analysis of PPCG Embeddings

In this section we look at prediction-attention-weighted scores for PPCG in relation to tissue morphology.

In [36]:
##############
## FUNCTION ##
##############

def sl(list_, indice_array):
    """
    """
    return [list_[i] for i in indice_array]

def grouped_stratified_kfold_split(labels, groups, num_folds=3):
    """
    """
    ## Data
    X = np.ones(labels.shape[0]) # dummy data

    ## Format labels
    labels = collapse_columns_2Darray_to_string(labels)

    ## Initiate
    sgkf = StratifiedGroupKFold(n_splits=num_folds)

    ## Split 
    sgkf.get_n_splits(X, labels)

    ## Get indices
    train_folds = []
    test_folds = []
    for i, (train_index, test_index) in enumerate(sgkf.split(X, labels, groups)):
        train_folds += [train_index]
        test_folds += [test_index]

    ## End
    return train_folds, test_folds

def collapse_columns_2Darray_to_string(arr, sep=""):
    """
    """
    arr_str = []
    for arr_ in arr:
        arr_ = [arr__ for arr__ in arr_.astype(str)]
        arr_ = sep.join(arr_)
        arr_str += [arr_]
    return np.array(arr_str)


#
###

In [37]:
##############
## INITIATE ##
##############

## Imports 
import os
import numpy as np
import pickle
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import umap
import seaborn as sns

## Import modules
from pawmil.f_utils import format_path_file
from pawmil.f_utils import format_label_file

## Paths
pwd = "/Users/willembonnaffe/Library/CloudStorage/OneDrive-Nexus365/BDI/projects/PAWMIL-2D/b2_2_2/"

#
###

In [38]:
#########################
## USER DEFINED INPUTS ##z
#########################

## Parameters
pt_folder = pwd + "PPCG_MQ_SmallMedium_Nozero_256_MultiTask/" 
selected_testing_fold = 0
selected_training_fold = 0
device = "mps"

## Paths
pt_subfolder_out = pt_folder + f'hypertrain_fold{selected_testing_fold}/'
pt_embeddings_in = pt_folder + 'encoder/obj_embeddings.pkl'
pt_label_file = pt_folder + "tab_labels.txt" 
pt_groups_file = pt_folder + "tab_groups.txt"
pt_groups_2_file = pt_folder + "tab_groups_2.txt"
pt_models_folder = pt_subfolder_out + "models/"

## Fixed parameters
num_folds = 5
num_hidden = 256
num_repeats = 3

#
###

In [39]:
##############
## INITIATE ##
##############

## Get labels and groups and embeddings
labels = format_label_file(pt_label_file)
groups = np.genfromtxt(pt_groups_file, dtype="str")
groups_2 = np.genfromtxt(pt_groups_2_file, dtype="str")
embeddings = pickle.load(open(pt_embeddings_in, "rb"))

## Dependent parameters
num_embeddings = embeddings[0].shape[1]

#
###

In [40]:
###########################
## PREPARE TRAINING DATA ##
###########################

from sklearn.model_selection import StratifiedGroupKFold

## Determine test fold indices 
trainval_folds, test_folds = grouped_stratified_kfold_split(labels, groups, num_folds)
trainval_index = trainval_folds[selected_testing_fold-1]
test_index = test_folds[selected_testing_fold-1]

## Determine validation fold indices
train_folds, val_folds = grouped_stratified_kfold_split(labels[trainval_index], groups[trainval_index], num_folds)
train_index = train_folds[selected_training_fold]
val_index = val_folds[selected_training_fold]

## Split data in folds
train_labels, val_labels, test_labels = sl(sl(labels, trainval_index), train_index), sl(sl(labels, trainval_index), val_index), sl(labels, test_index)
train_embeddings, val_embeddings, test_embeddings = sl(sl(embeddings, trainval_index), train_index), sl(sl(embeddings, trainval_index), val_index), sl(embeddings, test_index)

#
###

/Users/willembonnaffe/anaconda3/envs/pytorch/lib/python3.8/site-packages/sklearn/model_selection/_split.py:876: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/Users/willembonnaffe/anaconda3/envs/pytorch/lib/python3.8/site-packages/sklearn/model_selection/_split.py:876: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [49]:
######################################
## CHECK TRAIN VALIDATION TEST SETS ##
######################################

## Check
# print(np.array(train_labels).sum(0))
print(np.array(test_labels).sum(0))
print(np.array(test_labels)[np.argwhere(np.array(groups_2)[test_index] == 'CRUK')].sum(0))
print(np.array(test_labels)[np.argwhere(np.array(groups_2)[test_index] == 'EOPC')].sum(0))
# print(groups[train_index])
# print(groups_2[train_index])
# print(groups_2[test_index])
# print(groups_2[np.argwhere(groups_2[test_index] == 'CRUK')])
#
###

[ 7. 12. 42. 32.  5.  8. 18. 39. 11. 17.  7. 26. 25. 17. 19. 14. 24.]
[[ 5.  5. 25. 25.  1.  4.  5. 16.  1.  6.  1. 14. 11.  8.  5.  1.  3.]]
[[ 2.  3. 17.  5.  4.  4.  6. 16.  9.  8.  4. 11. 12.  8.  9. 10. 13.]]
